# Skills: Claude limpia la ENAHO

Bitácora de la sesión: cada paso del proyecto queda registrado aquí.

## Paso 1 · Preparar (minutos 0–8)

Comandos corridos en la terminal, desde la raíz del repositorio:

```bash
gh repo create diplomado-python-skills --add-readme --clone
uv init --bare --name diplomado-python-skills
uv add pandas pyarrow geopandas matplotlib
uv add --dev ipykernel

mkdir -p datos && cd datos
curl -fLO https://proyectos.inei.gob.pe/iinei/srienaho/descarga/STATA/966-Modulo34.zip
curl -fLO https://proyectos.inei.gob.pe/iinei/srienaho/descarga/STATA/966-Modulo02.zip
curl -fLO https://raw.githubusercontent.com/d2cml-ai/Data-Science-Python/main/sessions/06-geoespacial/lab/Data/peru_departamental_simple.geojson
```

`datos/` está en `.gitignore`: los datos crudos se descargan, no se versionan.

### Descomprimir los módulos

In [1]:
import zipfile
from pathlib import Path

import geopandas as gpd
import pandas as pd

DATOS = Path("datos")

for modulo in ["966-Modulo34", "966-Modulo02"]:
    with zipfile.ZipFile(DATOS / f"{modulo}.zip") as z:
        z.extractall(DATOS)

sorted(p.name for p in DATOS.glob("*/*.dta"))

['enaho01-2024-200.dta',
 'enaho_tabla_ciuo_88.dta',
 'enaho_tabla_cno_2015.dta',
 'sumaria-2024-12g.dta',
 'sumaria-2024.dta']

### Verificar que los datos coinciden con la clase

Se esperan 33,691 hogares en la sumaria, 117,721 filas en miembros y 25 departamentos.

In [2]:
sumaria = pd.read_stata(DATOS / "966-Modulo34/sumaria-2024.dta", convert_categoricals=False)
miembros = pd.read_stata(DATOS / "966-Modulo02/enaho01-2024-200.dta", convert_categoricals=False)
departamentos = gpd.read_file(DATOS / "peru_departamental_simple.geojson")

print("sumaria:", sumaria.shape)
print("miembros:", miembros.shape)
print("departamentos:", departamentos.shape, list(departamentos.columns))

assert len(sumaria) == 33_691
assert len(miembros) == 117_721
assert len(departamentos) == 25

sumaria: (33691, 163)
miembros: (117721, 40)
departamentos: (25, 5) ['NOMBDEP', 'COUNT', 'FIRST_IDDP', 'HECTARES', 'geometry']


## Paso 2 · Limpiar (minutos 8–20)

La skill vive en `.claude/skills/limpiando-enaho/`:

- `SKILL.md`: qué hace y cuándo usarla. Claude solo lee `name` y `description` para decidir si la activa.
- `reference.md`: variables y métodos de ponderación; se lee solo si hace falta.
- `scripts/limpiar.py`: los siete pasos, siempre en el mismo orden.

Lo que no se discute va en el script (llaves, filtro, unión, validación). El criterio va en el texto.

In [3]:
!uv run .claude/skills/limpiando-enaho/scripts/limpiar.py --anio 2024

ENAHO 2024: validación
Hogares en la sumaria: 33,691
Residentes habituales: 108,664 de 117,721 filas en miembros
Pobreza (sumaria, factor07 × mieperho): 27.58 %
Pobreza extrema (sumaria, factor07 × mieperho): 5.50 %
Pobreza (base de personas, facpob07): 27.53 %

OK    llaves de hogar únicas
OK    llaves de persona únicas
OK    ninguna fila perdida ni duplicada
OK    toda persona encontró su hogar
OK    ubigeo con 6 dígitos
OK    residentes = 108,664
OK    pobreza a 0.1 de 27.6 %
OK    extrema a 0.1 de 5.5 %



Guardado: salidas/personas_2024.parquet


### Revisar contra las referencias de la clase

Se esperan Cusco 18.5 %, urbano 24.8 %, rural 39.3 %, Cajamarca 45.0 % e Ica 6.0 %, calculadas con la sumaria (`factor07 × mieperho`). Con `facpob07` pueden moverse unas décimas.

In [4]:
base = pd.read_parquet("salidas/personas_2024.parquet")

def pobreza_pct(df):
    return round(100 * df.loc[df["pobre"], "facpob07"].sum() / df["facpob07"].sum(), 2)

{
    "Cusco": pobreza_pct(base[base["departamento"] == "08"]),
    "Urbano": pobreza_pct(base[base["urbano"]]),
    "Rural": pobreza_pct(base[~base["urbano"]]),
    "Cajamarca": pobreza_pct(base[base["departamento"] == "06"]),
    "Ica": pobreza_pct(base[base["departamento"] == "11"]),
}

{'Cusco': np.float32(18.54),
 'Urbano': np.float32(24.74),
 'Rural': np.float32(39.44),
 'Cajamarca': np.float32(44.99),
 'Ica': np.float32(5.91)}

## Paso 3 · Graficar (minutos 20–32)

`.claude/skills/graficando-enaho/` encadena con `limpiando-enaho` si falta la base. Las reglas fijas viven en `scripts/graficar.py`: `facpob07`, barras desde cero, un color, valor en cada barra, título, fuente y PNG a 200 dpi en `figuras/`. Área y sexo se ordenan por valor; la edad conserva su orden natural.

In [5]:
!for corte in area sexo edad; do uv run .claude/skills/graficando-enaho/scripts/graficar.py --por $corte 2>/dev/null; done

urbano
Rural     39.44
Urbano    24.74
Guardado: figuras/pobreza_area_2024.png


p207
Mujer     27.68
Hombre    27.37
Guardado: figuras/pobreza_sexo_2024.png


p208a
0 a 17      36.47
18 a 29     23.95
30 a 44     27.87
45 a 59     21.41
60 o más    22.02
Guardado: figuras/pobreza_edad_2024.png


## Paso 4 · Mapear (minutos 20–32)

`.claude/skills/mapeando-enaho/` une `ubigeo[:2]` con `FIRST_IDDP`, exige 25 departamentos con dato y guarda `mapas/pobreza_departamento.png`. Pondera con `factor07 × mieperho` para replicar las cifras de la clase: Cajamarca 45.0 %, Loreto 43.0 %, Puno 39.9 %, Ica 6.0 %, Moquegua 11.0 % y Madre de Dios 11.1 %.

In [6]:
!uv run .claude/skills/mapeando-enaho/scripts/mapear.py 2>/dev/null

Más pobreza:
CAJAMARCA 45.0
   LORETO 43.0
     PUNO 39.9
Menos pobreza:
          ICA  6.0
     MOQUEGUA 11.0
MADRE DE DIOS 11.1
Guardado: mapas/pobreza_departamento.png


## Paso 5 · Evaluar (minutos 32–45)

Se corrieron tres pedidos con y sin skills con `claude -p`, en copias limpias del repositorio. Los resultados están en `evaluacion.md`.

La evaluación encontró un error: `$0` en `limpiando-enaho` tomaba una palabra del pedido como año. Se corrigió fijando `--anio 2024` y se repitieron las corridas con skills.